In [1]:
# Import the required PySpark modules
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from pyspark.sql.functions import sum

# ---------------------------------------------------------
# Step 1: Initialize the Spark Session
# ---------------------------------------------------------

In [5]:
spark = SparkSession.builder \
    .appName("Mohammed's Assignment") \
    .master("local[*]") \
    .getOrCreate()

# ---------------------------------------------------------
# Step 2: Define the Schema
# ---------------------------------------------------------

In [ ]:
flight_schema = StructType([
    StructField("DEST_COUNTRY_NAME", StringType(), True),
    StructField("ORIGIN_COUNTRY_NAME", StringType(), True),
    StructField("count", IntegerType(), True)
])

# ---------------------------------------------------------
# Step 3: Read the Stream from the Directory
# ---------------------------------------------------------

In [7]:
# Using "file://" forces Spark to look at the local container 
# filesystem instead of the default Hadoop/HDFS filesystem.
streaming_df = spark.readStream \
    .schema(flight_schema) \
    .option("header", "true") \
    .csv("file:///data/csv/")

# ---------------------------------------------------------
# Step 4: Aggregate the Data
# ---------------------------------------------------------

In [8]:
# Group by destination and origin, then sum the "count" column
# to get the total count for each country pair.
aggregated_df = streaming_df \
    .groupBy("DEST_COUNTRY_NAME", "ORIGIN_COUNTRY_NAME") \
    .agg(sum("count").alias("total_count"))

# ---------------------------------------------------------
# Step 5: Write the Output to Console
# ---------------------------------------------------------

In [9]:
# Using outputMode("update") guarantees that Spark only prints 
# the specific country pairs whose totals have changed in the 
# current micro-batch, fulfilling the assignment requirement.
query = aggregated_df.writeStream \
    .outputMode("update") \
    .format("console") \
    .start()

In [ ]:
# Keep the streaming application running
query.awaitTermination()